# 105 - Real-Time Streaming

`Client` includes a streaming client that delivers real-time
market data over a persistent connection. Streaming uses a **push-callback**
model: you open a streaming session, register a callback, and subscribe.
The SDK invokes your callback once per decoded event on a background
thread.

Each event is a typed object - `Quote`, `Trade`, `LoginSuccess`,
`MarketOpen`, and so on - that you narrow with `isinstance` or a
`match` statement. Fields are plain attributes (`event.bid`,
`event.price`, `event.ms_of_day`).

This notebook covers:

1. Opening a streaming session
2. Subscribing to AAPL quotes
3. Collecting events over a time window
4. Displaying quote updates in a table
5. Subscribing to trades
6. Trade-flow summary

**Requires an active streaming session.** Real-time streaming needs a
market data subscription with real-time access, and data flows only
during market hours (09:30 - 16:00 ET). The callback runs on the
event-dispatch thread - keep it fast and hand work off to the main
thread, as shown below.

In [ ]:
import time
import threading
from collections import deque

import pandas as pd
from datetime import datetime

from thetadatadx import (
    Credentials, Config, Client, Contract,
    Quote, Trade, LoginSuccess,
)

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. Open a Streaming Session

We keep a thread-safe buffer and append decoded events from the callback.
The callback fires on a background thread, so we guard the buffer with a
lock. `client.streaming(callback)` is a context manager: entering it opens
the connection, leaving it stops streaming and drains in-flight events.

In [ ]:
creds = Credentials.from_file("creds.txt")
config = Config.production()
tdx = Client(creds, config)

# Thread-safe event buffer shared between the callback thread and this cell.
_buffer = deque(maxlen=20_000)
_lock = threading.Lock()
_login = threading.Event()


def on_event(event):
    # Runs on the dispatch thread - keep it cheap.
    if isinstance(event, LoginSuccess):
        _login.set()
    with _lock:
        _buffer.append(event)


def drain():
    # Move buffered events into a list for analysis on the main thread.
    with _lock:
        items = list(_buffer)
        _buffer.clear()
    return items


# Open the streaming session and keep the handle for the cells below.
session = tdx.streaming(on_event)
session.__enter__()

# Wait for the login acknowledgement.
if _login.wait(timeout=5.0):
    print("Streaming session established.")
else:
    print("No login acknowledgement yet - the streaming feed may be unavailable.")
print(tdx)

## 2. Subscribe to AAPL Quotes

Build a typed `Contract` and subscribe to its quote stream. The server
acknowledges with a `ReqResponse` event, then streams `Quote` events
carrying the full NBBO (`bid`, `ask`, `bid_size`, `ask_size`,
`ms_of_day`).

In [ ]:
drain()  # clear any startup events
tdx.stream.subscribe(Contract.stock("AAPL").quote())
print("Subscribed to AAPL quotes.")

## 3. Collect Quote Events for 10 Seconds

In [ ]:
print("Collecting AAPL quotes for 10 seconds...")
time.sleep(10)

events = drain()
quote_events = [e for e in events if isinstance(e, Quote)]

quotes = [
    {
        "bid": e.bid,
        "ask": e.ask,
        "bid_size": e.bid_size,
        "ask_size": e.ask_size,
        "ms_of_day": e.ms_of_day,
        "symbol": e.contract.symbol,
    }
    for e in quote_events
]
print(f"Collected {len(quotes)} quote events in 10 seconds")

## 4. Display Quote Updates in a Table

In [ ]:
if quotes:
    df_quotes = pd.DataFrame(quotes)
    df_quotes["spread"] = df_quotes["ask"] - df_quotes["bid"]

    print(f"Quote event rate: {len(quotes) / 10:.1f} events/sec")
    print(f"Mean spread: ${df_quotes['spread'].mean():.4f}")
    print()
    display(df_quotes[["ms_of_day", "bid", "ask", "spread",
                       "bid_size", "ask_size"]].head(20))
else:
    print("No quote events received (market may be closed or feed unavailable).")

## 5. Subscribe to AAPL Trades

In [ ]:
tdx.stream.subscribe(Contract.stock("AAPL").trade())
print("Subscribed to AAPL trades.")

print("Collecting AAPL trades + quotes for 10 seconds...")
time.sleep(10)

events = drain()
trade_events = [e for e in events if isinstance(e, Trade)]
trades = [
    {
        "price": e.price,
        "size": e.size,
        "exchange": e.exchange,
        "condition": e.condition,
        "ms_of_day": e.ms_of_day,
        "symbol": e.contract.symbol,
    }
    for e in trade_events
]
other = len(events) - len(trades)
print(f"Trade events:  {len(trades)}")
print(f"Other events:  {other}")

## 6. Trade-Flow Summary

In [ ]:
if trades:
    df_trades = pd.DataFrame(trades)
    print(f"Trade event rate: {len(trades) / 10:.1f} events/sec")
    print(f"Total volume: {df_trades['size'].sum():,} shares")
    vwap = (df_trades["price"] * df_trades["size"]).sum() / df_trades["size"].sum()
    print(f"VWAP: ${vwap:.2f}")
    print()
    display(df_trades[["ms_of_day", "price", "size", "exchange", "condition"]].head(20))
else:
    print("No trade events received (market may be closed or feed unavailable).")

## 7. Close the Streaming Session

Leaving the streaming context (or calling `stop_streaming()` /
`shutdown()`) stops the background thread and drains in-flight events, so
the callback is guaranteed to have stopped firing before control returns.

In [ ]:
# Close the session opened in step 1. In normal use, prefer the
# `with tdx.streaming(on_event) as session:` form, which closes
# automatically on block exit.
session.__exit__(None, None, None)
print(tdx)
print("Clean shutdown complete.")

---

A more idiomatic pattern wraps the whole flow in a single `with` block:

```python
with tdx.streaming(on_event) as session:
    session.subscribe(Contract.stock("AAPL").quote())
    session.subscribe(Contract.stock("AAPL").trade())
    time.sleep(30)
# session is closed and drained here
```

**Next:** [106 - Live Option Chain](106_live_option_chain.ipynb)